# 0. INSTALACIÓN DE LIBRERÍAS

In [ ]:
# ============================================================
# 0. INSTALACIÓN DE LIBRERÍAS
# ============================================================
!pip install -q -U faiss-cpu sentence-transformers transformers accelerate bitsandbytes rank_bm25 pymupdf evaluate rouge_score langchain langchain-text-splitters

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 90.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.4/112.4 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.5/167.5 kB 17.3 MB/s eta 0:00:00


# 1. IMPORTACIÓN DE LIBRERÍAS

In [ ]:
# ============================================================
# 1. IMPORTACIÓN DE LIBRERÍAS
# ============================================================
import fitz  # PyMuPDF para el PDF legal
import re
import numpy as np
import pandas as pd
import torch
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
import faiss
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

In [ ]:
# ============================================================
# 2. CARGA DEL DOCUMENTO (CONTRATO HIPOTECARIO REAL)
# ============================================================
def extract_clean_legal_text(pdf_path):
    doc = fitz.open(pdf_path)
    full_text = []

    for page_num, page in enumerate(doc):
        # Extraemos el texto de la página
        page_text = page.get_text("text")

        # --- LIMPIEZA INICIAL ---
        # 1. Eliminamos números de página aislados al final o inicio (evita cortes en el ROUGE)
        page_text = re.sub(r'^\d+\s*\n|\n\s*\d+$', '', page_text)

        # 2. Corregimos palabras cortadas por saltos de línea (guiones al final de línea)
        page_text = re.sub(r'(\w)-\n(\w)', r'\1\2', page_text)

        # 3. Normalizamos espacios en blanco (evita que el splitter cuente mal)
        page_text = re.sub(r' +', ' ', page_text)

        # 4. Eliminar saltos de línea simples que rompen oraciones,
        # pero mantener los dobles saltos de línea (párrafos).
        page_text = re.sub(r'(?<!\n)\n(?!\n)', ' ', page_text)
        full_text.append(page_text)

    # Unimos con un marcador claro para el splitter
    return "\n\n".join(full_text)

pdf_path = "CONTRATO COMPRA DE DEUDA GAMARRA GARRO.pdf"
raw_legal_text = extract_clean_legal_text(pdf_path)

print(f"Caracteres extraídos: {len(raw_legal_text)}")

Caracteres extraídos: 72083


In [ ]:
# ============================================================
# 3. CHUNKING + OVERLAP (Parámetros: 512/50)
# ============================================================
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800, #1200,  #1000,    # Un poco más de espacio para párrafos legales largos
    chunk_overlap=150, #200, #50,   # Mucho más solapamiento para no perder el contexto
    length_function=len,
    is_separator_regex=False,
    separators=["\n\n", "\n", ". ", " ", ""] # Añadimos el punto con espacio ". " para no romper oraciones jurídicas
)

chunks = text_splitter.split_text(raw_legal_text)

# Verificación de calidad (Opcional pero recomendada)
print(f"Número total de chunks: {len(chunks)}")
print(f"Ejemplo del chunk 1:\n{chunks[0][:200]}...")

Número total de chunks: 126
Ejemplo del chunk 1:
Resolución SBS N°02929-2022  OFICIO Nº 41967-2022-SBS  Contrato de Préstamo Hipotecario (Compra de Deuda)  Señor Notario, sírvase extender en su registro de escritura públicas la presente minuta y sus...


In [ ]:
# ============================================================
# 4. EMBEDDINGS Y FAISS HNSW
# ============================================================
# 1. Generar Embeddings
# El modelo 'paraphrase-multilingual-MiniLM-L12-v2' es excelente para español
# 1. Definir e inicializar el modelo (Esto resuelve el NameError)
model_embed = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

embeddings = model_embed.encode(chunks)
embeddings = np.array(embeddings).astype('float32')

# 2. NORMALIZACIÓN (Paso clave para mejorar el Grounding)
# Esto convierte la búsqueda L2 en una búsqueda de Similitud de Coseno
faiss.normalize_L2(embeddings)

# 3. Configuración de FAISS HNSW
dimension = embeddings.shape[1]
# Usamos Metric_INNER_PRODUCT porque ya normalizamos (equivale a Coseno)
index = faiss.IndexHNSWFlat(dimension, 16, faiss.METRIC_INNER_PRODUCT)

index.hnsw.efConstruction = 100 # Subimos de 40 a 100 para mayor precisión en la construcción
index.add(embeddings)

# 4. Ajustar efSearch (Precisión en tiempo de búsqueda)
index.hnsw.efSearch = 64

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
print(dimension)

In [ ]:
query = "Cuando el banco podrá modificar las “tasas de interés”?"

# 1. Generar el embedding de la pregunta
query_embedding = model_embed.encode([query]).astype('float32')

# IMPORTANTE: No vuelvas a crear el índice. Usa el que ya tiene los 'chunks'.
# 2. Buscar en el índice ya existente
D, I = index.search(query_embedding, k=3)

# 3. Mostrar el resultado
print(f"Mejor fragmento encontrado: {chunks[I[0][0]]}")

Mejor fragmento encontrado: .  Las modificaciones al Contrato distintas a tasas de interés, tales como comisiones y gastos, que  representen un incremento respecto a lo pactado, serán informadas por INTERBANK al CLIENTE en  forma previa a su aplicación con una anticipación no menor de cuarenta y cinco (45) días calendario.  En cualquier caso, la comunicación indicará el momento a partir del cual la modificación entrará en  vigencia.  7.3  En caso corresponda, conforme a la Ley Aplicable, INTERBANK remitirá las modificaciones al  Cronograma de Pagos al domicilio del CLIENTE, con la anticipación prevista en la Ley Aplicable,  incluyendo el costo efectivo anual que corresponda por el saldo remanente de la operación crediticia,  que se identificará como “Costo Efectivo Anual Remanente”


In [ ]:
# ============================================================
# 5. RETRIEVAL HÍBRIDO (BM25 + FAISS)
# ============================================================
tokenized_corpus = [doc.lower().split() for doc in chunks]
bm25 = BM25Okapi(tokenized_corpus)

def hybrid_retrieval(query, k=5):
    # 1. FAISS Semantic Search (Normalizado)
    query_vec = model_embed.encode([query]).astype('float32')
    faiss.normalize_L2(query_vec) # Mantener la normalización que aplicamos antes
    distances, indices = index.search(query_vec, k)

    # 2. BM25 Keyword Search
    tokenized_query = query.lower().split()
    # Obtenemos los scores para poder rankear
    bm25_scores = bm25.get_scores(tokenized_query)
    top_bm25_indices = np.argsort(bm25_scores)[::-1][:k]

    # 3. Reciprocal Rank Fusion (RRF) - Combinación inteligente
    combined_scores = {}

    # RRF para FAISS
    for rank, idx in enumerate(indices[0]):
        combined_scores[idx] = combined_scores.get(idx, 0) + 1 / (rank + 60)

    # RRF para BM25
    for rank, idx in enumerate(top_bm25_indices):
        combined_scores[idx] = combined_scores.get(idx, 0) + 1 / (rank + 60)

    # Ordenar por el score combinado y tomar los mejores K
    sorted_indices = sorted(combined_scores.items(), key=lambda x: x[1], reverse=True)[:k]

    final_context = [chunks[idx] for idx, score in sorted_indices]
    return final_context

In [ ]:
# ============================================================
# 6. RERANKER (CROSS-ENCODER)
# ============================================================
from sentence_transformers import CrossEncoder

# Usamos el modelo BGE-Reranker-V2-M3 (Multilingüe)
reranker = CrossEncoder('BAAI/bge-reranker-v2-m3')

def get_reranked_context(query, documents, threshold=-2.0):
    if not documents:
        return []

    # 1. Crear pares [pregunta, documento]
    pairs = [[query, doc] for doc in documents]

    # 2. Obtener scores (este modelo devuelve valores donde más alto es mejor)
    scores = reranker.predict(pairs)

    # 3. Ordenar y filtrar por relevancia
    scored_docs = sorted(zip(scores, documents), key=lambda x: x[0], reverse=True)

    # Threshold de -2.0 es un buen punto de partida para BGE
    ranked_docs = [doc for score, doc in scored_docs if score > threshold]

    return ranked_docs[:4]

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

In [ ]:
reranker

In [ ]:
# ============================================================
# 7. LLM QWEN + GENERACIÓN CON CITAS (CITATION PER SENTENCE)
# ============================================================

# 1. Configuración del Pipeline (Aseguramos el manejo de device)
model_id = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
# Usamos torch_dtype para evitar errores de precisión
llm_pipeline = pipeline(
    "text-generation",
    model=model_id,
    device=0,
    torch_dtype=torch.float16,
    model_kwargs={"low_cpu_mem_usage": True}
)

def generate_rag_response(query):
    # Recuperación optimizada (bloques anteriores)
    raw_context = hybrid_retrieval(query)
    final_context = get_reranked_context(query, raw_context)

    if not final_context:
        return "Información no disponible en el contrato.", []

    # Formatear contexto con índices
    context_str = "\n".join([f"[{i+1}] {doc}" for i, doc in enumerate(final_context)])

    # Prompt en formato ChatML (Nativo de Qwen)
    prompt = f"""<|im_start|>system
  Eres un experto legal riguroso. Responde usando exclusivamente el contexto. PROHIBIDO INFERIR O USAR CONOCIMIENTO PREVIO. Si la respuesta exacta no está, di: 'Información no disponible' y no agregues nada más.<|im_end|>
  <|im_start|>user
  Contexto:
  {context_str}

  Pregunta: {query}<|im_end|>
  <|im_start|>assistant
  """

    # Generación determinista (do_sample=False para que no invente)
    outputs = llm_pipeline(
        prompt,
        max_new_tokens=400,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    # Extraer solo la respuesta del asistente
    full_output = outputs[0]['generated_text']
    respuesta = full_output.split("<|im_start|>assistant")[-1].strip()

    return respuesta, final_context

# --- FUNCIÓN PARA SUBIR EL ROUGE-L ---
def clean_for_eval(text):
    """Limpia la respuesta de la IA para que coincida con la referencia humana."""
    # 1. Eliminar citaciones [1], [2,3], etc.
    text = re.sub(r'\[\d+(?:,\s*\d+)*\]', '', text)
    # 2. Eliminar saltos de línea y espacios extra
    text = " ".join(text.split())
    return text.lower()

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
# ============================================================
# 8. EVALUACIÓN: GROUNDING SCORE + ROUGE
# ============================================================
import evaluate
# Cargamos la métrica ROUGE
rouge_metric = evaluate.load('rouge')

def clean_text_for_metric(text):
    """Limpia citaciones y ruidos para una comparación justa."""
    if not text: return ""
    # Eliminar citaciones tipo [1], [1,2], [1][2]
    text = re.sub(r'\[\d+(?:,\s*\d+)*\]', '', text)
    # Eliminar puntuación innecesaria y pasar a minúsculas
    text = re.sub(r'[^\w\s]', '', text.lower())
    return " ".join(text.split())

def calculate_advanced_metrics(query, ai_response, reference_response, context_list):
    # EXCEPCIÓN: Si la IA declara correctamente que no hay información
    if "información no disponible" in ai_response.lower():
        # Si realmente no había contexto relevante, el score es 1.0
        return {"rouge_l": 1.0 if "disponible" in reference_response.lower() else 0.0,
                "grounding": 1.0}

    # 1. Limpieza de textos para ROUGE (Crucial para subir de 0.06)
    clean_ai = clean_text_for_metric(ai_response)
    clean_ref = clean_text_for_metric(reference_response)

    # 2. Calcular ROUGE-L
    rouge_results = rouge_metric.compute(
        predictions=[clean_ai],
        references=[clean_ref],
        rouge_types=['rougeL']
    )
    rouge_l_score = round(rouge_results['rougeL'], 4)

    # 3. Grounding Score (Fidelidad al contexto)
    # Filtramos 'stopwords' comunes para que el score sea sobre contenido real
    stopwords = {'el', 'la', 'los', 'las', 'un', 'una', 'en', 'de', 'del', 'que', 'y', 'a', 'con'}

    ai_words = set(clean_ai.split()) - stopwords
    context_text = clean_text_for_metric(" ".join(context_list))
    context_words = set(context_text.split())

    if not ai_words:
        grounding_score = 0.0
    else:
        # Qué porcentaje de las palabras clave de la IA están realmente en el contrato
        intersection = ai_words.intersection(context_words)
        grounding_score = len(intersection) / len(ai_words)

    return {
        "rouge_l": rouge_l_score,
        "grounding": round(grounding_score, 2)
    }

In [ ]:
# ============================================================
# 9. DEMO
# ============================================================

pregunta = "¿Cuándo INTERBANK hara modificaciones al cronograma de pagos?"
respuesta, contextos = generate_rag_response(pregunta)
metrics = calculate_advanced_metrics(query, res_ia, pregunta,  contextos)
grounding = metrics['grounding']
rouge_1 = metrics['rouge_l']

print(f"--- PREGUNTA ---\n{pregunta}")
print(f"\n--- RESPUESTA (CON CITAS) ---\n{respuesta}")
print(f"\n--- MÉTRICAS ---")
print(f"Grounding Score (Confianza): {grounding}")
print(f"rouge_1 Score (Confianza): {rouge_1}")

Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- PREGUNTA ---
¿Cuándo INTERBANK hara modificaciones al cronograma de pagos?

--- RESPUESTA (CON CITAS) ---
INTERBANK hará modificaciones al cronograma de pagos cuando se cumpla uno de los siguientes criterios:

1. Se informe a INTERBANK por parte del CLIENTE que las modificaciones representan un incremento respecto a lo pactado.

2. La comunicación indica el momento a partir del cual la modificación entrará en vigor.

3. INTERBANK remita las modificaciones al Cronograma de Pagos al domicilio del CLIENTE, siguiendo la anticipación prevista en la Ley Aplicable.

4. INTERBANK procederá a la reducción del número de cuotas si no se ha hecho antes.

5. INTERBANK entregará el cronograma de pagos modificado dentro de los 7 días calendarios posteriores a la solicitud del CLIENTE.

6. INTERBANK aceptará el pago anticipado sin pagar penalidad alguna.

7. INTERBANK procesará la solicitud del CLIENTE para cambiar la moneda del financiamiento, siempre que el CLIENTE esté al día en sus obligacione